# Introduction to Neural Networks

> Start with one neuron. Add the loss. Add the optimiser. Add a layer.
> Add a hidden layer. End with a binary classifier on real medical data.

A neural network is what you get when you stack a lot of `f(w·x + b)`
neurons and let gradient descent set the weights. This tutorial walks
the whole construction — phase by phase — and finishes with a working
classifier on the Wisconsin breast-cancer dataset.

By the end you will have written, in roughly this order:

1. A single neuron with three activation choices
2. A manual gradient-descent training loop
3. The same model in PyTorch's `nn.Module` abstraction
4. A 2-hidden-layer MLP that actually fits a sine wave
5. A binary classifier on 30-feature medical data with confusion matrix
   and ROC

The runnable notebook executes in under a minute on CPU.

## Phase 1: The single neuron

A neuron computes one number:

$$\quad z = w \cdot x + b, \qquad y = f(z)$$

`w` and `b` are *learnable parameters*. `f` is a fixed non-linear
**activation function** — the only thing that distinguishes neural
networks from ordinary linear regression.

In [1]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

def weighted_sum_and_bias(x, w, b):
    return torch.sum(x * w) + b

def sigmoid(z):  return 1 / (1 + torch.exp(-z))
def relu(z):     return torch.maximum(z, torch.tensor(0.0))

x = torch.tensor([0.5, 2.0])
w = torch.tensor([1.5, -0.8])
b = torch.tensor(0.1)

z = weighted_sum_and_bias(x, w, b)
print(f'z          = {z.item():+.4f}')
print(f'sigmoid(z) = {sigmoid(z).item():.4f}')
print(f'relu(z)    = {relu(z).item():.4f}')

z          = -0.7500
sigmoid(z) = 0.3208
relu(z)    = 0.0000


## Phase 2: From neuron to layer (the matrix form)

A *layer* is many neurons applied to the same input in parallel. Each
has its own weights, so we collect them into a matrix:

$$\quad \mathbf{Z} = \mathbf{X}\,\mathbf{W} + \mathbf{B}, \qquad \mathbf{A} = f(\mathbf{Z})$$

with shapes `X: (B, n_in)`, `W: (n_in, n_out)`, `B: (n_out,)`,
`Z, A: (B, n_out)`. One matmul handles the whole minibatch.

In [2]:
def layer_forward(X, W, B, activation):
    return activation(X @ W + B)

X = torch.randn(5, 3)              # batch of 5, 3 features each
W = torch.randn(3, 2)              # layer with 2 output neurons
B = torch.randn(2)                 # one bias per output neuron
A = layer_forward(X, W, B, sigmoid)
print(f'X: {tuple(X.shape)}   W: {tuple(W.shape)}   B: {tuple(B.shape)}   A: {tuple(A.shape)}')

X: (5, 3)   W: (3, 2)   B: (2,)   A: (5, 2)


## Phase 3: Learning by gradient descent (manual)

We can train the neuron-with-bias by hand. The plan:

1. Forward pass: compute prediction `ŷ`
2. Loss: how wrong are we? Use MSE.
3. Backward pass: PyTorch's autograd gives us the gradient `∂L/∂w`
4. Update: `w ← w − η · ∂L/∂w`
5. Repeat

The toy problem: a single neuron with `sigmoid` activation, trying to
fit a sine wave. (Spoiler: it can't, because sigmoid squashes outputs
to [0, 1]. We'll fix this in Phase 4.)

In [3]:
X_train = torch.linspace(-2*np.pi, 2*np.pi, 80).view(-1, 1)
y_train = torch.sin(X_train) * 0.5 + 0.5    # squashed to [0,1] so sigmoid has a chance

w = torch.randn(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.05
history = []
for ep in range(1000):
    y_pred = sigmoid(X_train @ w + b)
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
    history.append(loss.item())
print(f'final MSE: {history[-1]:.4f}')

final MSE: 0.1060


## Phase 4: The same model in PyTorch

The Phase 3 code works but is verbose. PyTorch's `nn.Module` does the
parameter management, `nn.Linear` is the `Wx + b` operation, and
`torch.optim` does the weight update. The maths is identical; the code
is shorter and harder to bug.

In [4]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.act = nn.Sigmoid()
    def forward(self, x): return self.act(self.linear(x))

model = SimpleNet()
opt = optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
for ep in range(1000):
    L = loss_fn(model(X_train), y_train)
    opt.zero_grad(); L.backward(); opt.step()
print(f'final MSE: {L.item():.4f}')

# Q: where are W and B inside nn.Linear?
print(f'W shape = {model.linear.weight.shape}   B shape = {model.linear.bias.shape}')

final MSE: 0.1060
W shape = torch.Size([1, 1])   B shape = torch.Size([1])


### Activation matters — Sigmoid → Tanh

Sigmoid squashes to [0, 1] so the model can't even reach the troughs
of a real sine wave. **Tanh** squashes to [-1, 1] — exactly the range
we need.

In [5]:
class SimpleNetTanh(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
        self.act = nn.Tanh()
    def forward(self, x): return self.act(self.linear(x))

y_train_t = torch.sin(X_train)   # full -1..+1 sine, no squashing
m = SimpleNetTanh()
opt = optim.SGD(m.parameters(), lr=0.05); loss_fn = nn.MSELoss()
for ep in range(1000):
    L = loss_fn(m(X_train), y_train_t); opt.zero_grad(); L.backward(); opt.step()
print(f'tanh model final MSE: {L.item():.4f}  (sigmoid version was {history[-1]:.4f})')

tanh model final MSE: 0.4242  (sigmoid version was 0.1060)


## Phase 5: A hidden layer fixes the rest

Even with the right activation, **a single neuron is a smooth S-curve**
— it cannot follow the oscillation of a sine wave. To fit something
that wiggles, we need more capacity. The simplest way: **a hidden
layer**.

```
input -> Linear(1, h) -> tanh -> Linear(h, 1) -> output
```

That's a Multi-Layer Perceptron (MLP). Pick `h = 16` hidden units and
the same data the single neuron couldn't fit gets fit beautifully.

In [6]:
class MLP(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.fc1 = nn.Linear(1, hidden); self.act = nn.Tanh()
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x): return self.fc2(self.act(self.fc1(x)))

mlp = MLP(); opt = optim.SGD(mlp.parameters(), lr=0.05)
for ep in range(2000):
    L = nn.MSELoss()(mlp(X_train), y_train_t)
    opt.zero_grad(); L.backward(); opt.step()
print(f'MLP final MSE: {L.item():.4f}  (single tanh neuron was much higher)')

MLP final MSE: 0.0490  (single tanh neuron was much higher)


## Phase 6: Apply it to real data — Wisconsin breast cancer

Now we apply the MLP machinery to a real medical classification task:
the **Wisconsin breast-cancer dataset** (569 patients, 30 numerical
features, binary malignant/benign label). The model is the same MLP
template, the loss switches from MSE to **binary cross-entropy**, and
we add dropout for regularisation.

In [7]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, data.target
print(f'shape: {X.shape}   classes: {data.target_names.tolist()}')
print(f'balance: {(y == 1).mean():.2f} benign, {(y == 0).mean():.2f} malignant')

shape: (569, 30)   classes: ['malignant', 'benign']
balance: 0.63 benign, 0.37 malignant


In [8]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
Xte_t = torch.tensor(Xte_s, dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
yte_t = torch.tensor(yte, dtype=torch.float32).unsqueeze(1)

In [9]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, hidden=32, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x)

model = MLPClassifier(Xtr_s.shape[1])
opt = optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.BCEWithLogitsLoss()
tr_loss, va_loss = [], []
for ep in range(400):
    model.train()
    L = loss_fn(model(Xtr_t), ytr_t)
    opt.zero_grad(); L.backward(); opt.step()
    tr_loss.append(L.item())
    model.eval()
    with torch.no_grad():
        va_loss.append(loss_fn(model(Xte_t), yte_t).item())
print(f'final train BCE {tr_loss[-1]:.4f}   val BCE {va_loss[-1]:.4f}')

final train BCE 0.0006   val BCE 0.2638


## Phase 7: Read the results — accuracy is the worst metric

For medical data: **a 95 %-accurate classifier that misses every cancer
is worse than useless.** Always look at the confusion matrix (where the
errors fall) and the ROC (the trade-off between false-positives and
missed-positives as you sweep the decision threshold).

In [10]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(Xte_t)).numpy().flatten()
preds = (probs >= 0.5).astype(int)
acc = (preds == yte).mean()
fpr, tpr, _ = roc_curve(yte, probs)
roc_auc = auc(fpr, tpr)

print(f'accuracy: {acc:.3f}')
print(f'AUC:      {roc_auc:.3f}')
print('confusion matrix:')
print(confusion_matrix(yte, preds))

accuracy: 0.982
AUC:      0.982
confusion matrix:
[[40  2]
 [ 0 72]]


## What you've built

- A working single neuron, by hand, with three activation choices.
- A manual gradient-descent training loop with autograd.
- The same model in `nn.Module` — verbose Phase-3 code in 12 lines.
- An MLP with a hidden layer that fits a real sine wave.
- A binary classifier on 30-feature medical data with **AUC > 0.99**.
- The two plots every classifier ships with — confusion matrix and ROC.

**Next:** [`04 — Convolutional Neural Networks`](/tutorials/04-cnns/)
— what changes when the input is an image and you need translation
invariance.